# Homework 11 - Structure And Next Route

目标：把 micrograd 的文件结构和后续 PyTorch/D2L/LLM 路线接起来。这里仍然要用代码检查，不写空泛计划。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check

## 完整例子 - 用代码读项目结构

In [ ]:
for path in ['micrograd/engine.py', 'micrograd/nn.py', 'demo.ipynb', 'course/08_pytorch_bridge/homework.ipynb']:
    p = ROOT / path
    print(path, 'exists =', p.exists(), 'size =', p.stat().st_size if p.exists() else 0)

## 作业 1 - 文件职责映射

In [ ]:
# 填写说明：
# - 完成 student_inspect_micrograd_files，直接检查项目里三个入口是否存在。
# - 返回顺序固定：engine_has_value、nn_has_mlp、demo_exists。
# - 运行本 cell，看 qa_check 的提示。

def student_inspect_micrograd_files():
    # TODO: return engine_has_value, nn_has_mlp, demo_exists
    pass


qa_check('qa_check_11_file_roles', globals(), student_inspect_micrograd_files)


<details><summary>Show / Hide 本题提示</summary>

- 用 `Path` 检查文件，用 `hasattr` 检查模块里有没有对应类。
- 这题是“看项目结构”，不是背文件名。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_inspect_micrograd_files():
    import micrograd.engine as engine
    import micrograd.nn as nn
    return hasattr(engine, 'Value'), hasattr(nn, 'MLP'), (ROOT / 'demo.ipynb').exists()
```

</details>


## 作业 2 - micrograd 到 PyTorch 映射

In [ ]:
# 填写说明：
# - 这题拆成三步：micrograd 一步、PyTorch 一步、最后对比。
# - student_micrograd_bridge_step 返回顺序：grad、updated_w。
# - student_torch_bridge_step 返回顺序：grad、updated_w。
# - student_torch_bridge_probe 调用前两个函数。
# - 运行本 cell，看 qa_check 的提示。

def student_micrograd_bridge_step():
    # TODO: w=2, loss=(w*3-7)^2，backward 后更新 w，return grad, updated_w。
    pass


def student_torch_bridge_step():
    try:
        import torch
    except Exception:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO: 参考 student_micrograd_bridge_step，用 PyTorch 做同一件事。
    pass


def student_torch_bridge_probe():
    # TODO: 调用前两个函数，return micrograd_result, torch_result
    pass


qa_check('qa_check_11_torch_mapping', globals(), student_torch_bridge_probe)


<details><summary>Show / Hide 本题提示</summary>

- 和第 08 节一样，用 `w=2, x=3, y=7, loss=(w*x-y)^2`。
- 两套工具的梯度和更新结果应该一致。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_micrograd_bridge_step():
    w = Value(2.0)
    loss = (w * 3 - 7) ** 2
    loss.backward()
    grad = w.grad
    w.data -= 0.1 * w.grad
    return grad, w.data


def student_torch_bridge_step():
    import torch
    w = torch.tensor(2.0, requires_grad=True)
    loss = (w * 3 - 7) ** 2
    loss.backward()
    grad = w.grad.item()
    with torch.no_grad():
        w -= 0.1 * w.grad
    return grad, w.item()


def student_torch_bridge_probe():
    return student_micrograd_bridge_step(), student_torch_bridge_step()
```

</details>


## Modify - 从 micrograd 映射到 PyTorch

参考 `Value -> Tensor`，补一个训练循环动作的映射。

In [ ]:
# 填写说明：
# - 完成 student_update_bridge_demo，比较 micrograd 手动更新和 optimizer.step 的角色。
# - 运行本 cell，看 qa_check 的提示。

def student_update_bridge_demo():
    try:
        import torch
    except Exception:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO: 返回 manual_updated_w 和 optimizer_updated_w。
    pass


qa_check('qa_check_11_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- micrograd 里你自己写 `p.data -= lr * p.grad`。
- PyTorch 里常把这一步交给 optimizer；这里用 `torch.optim.SGD` 跑同一个数。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_update_bridge_demo():
    import torch
    manual_w = Value(2.0)
    manual_loss = (manual_w * 3 - 7) ** 2
    manual_loss.backward()
    manual_w.data -= 0.1 * manual_w.grad

    torch_w = torch.tensor(2.0, requires_grad=True)
    opt = torch.optim.SGD([torch_w], lr=0.1)
    torch_loss = (torch_w * 3 - 7) ** 2
    torch_loss.backward()
    opt.step()
    return manual_w.data, torch_w.item()
```

</details>


## 作业 3 - 下一阶段路线必须写成过关问题

In [ ]:
# 填写说明：
# - 填一个列表，列表里每项是字典；每个阶段必须有 pass_question。
# - 填完后运行本 cell，看 qa_check 的提示。

student_next_route = None
# 例子形状：
# [
#   {'stage':'PyTorch', 'pass_question':'...'},
#   {'stage':'D2L', 'pass_question':'...'},
# ]



qa_check('qa_check_11_next_route', globals(), student_next_route)

<details><summary>Show / Hide 本题提示</summary>

- 路线不是资源清单；每一站都要写“做成什么才算过关”。
- `pass_question` 应该能检查理解，比如能不能手算/跑通 backward、attention、generate。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
student_next_route = [
    {'stage': 'PyTorch', 'pass_question': '能不能用 Tensor 跑出 loss.backward 后的 grad？'},
    {'stage': 'D2L', 'pass_question': '能不能解释 MLP/优化器/Attention 在训练链路里的位置？'},
    {'stage': 'Happy-LLM', 'pass_question': '能不能讲清 Transformer 从 token 到 logits 的流程？'},
    {'stage': 'LLMs-from-scratch', 'pass_question': '能不能手写 GPT 的 forward/loss/generate 流程？'},
    {'stage': 'nanoGPT', 'pass_question': '能不能解释数据加载、checkpoint、混合精度和训练循环？'},
]
```

</details>


## Debug Lab - 路线学习的两个坑

In [ ]:
# 填写说明：
# - 完成 student_route_debug，判断“只会 import”和“资源堆太多”的问题。
# - 返回顺序固定：import_only_passes、too_many_resources、has_pass_questions。
# - 运行本 cell，看 qa_check 的提示。

def student_route_debug():
    # TODO: return import_only_passes, too_many_resources, has_pass_questions
    pass


qa_check('qa_check_11_debug', globals())


<details><summary>Show / Hide 本题提示</summary>

- 只会 `import torch` 不代表理解训练。
- 路线过载的修法是给每个阶段配一个过关问题，并少量推进。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_route_debug():
    return False, True, False
```

</details>


## 提交清单

- [ ] 文件职责映射通过
- [ ] PyTorch 映射通过
- [ ] 下一阶段路线通过
- [ ] Debug Lab 通过
- [ ] 我能解释 micrograd 学完后，为什么下一步必须补 PyTorch

<details><summary>Show / Hide 提示</summary>

先找完整例子的形状，再只改一个地方。填 TODO 前，先用小数字在纸上算一遍；测试失败时先判断错在数学、Python、计算图还是训练循环。

</details>

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>